# JAX DR Bridge Estimator — CUDA GPU Sweep

GPU-accelerated Doubly Robust Bridge Estimator for the Navegador gamma-surface.

This notebook runs the full construct-level DR sweep on CUDA GPU using JAX.
It is a self-contained port of `jax_dr_bridge.py` optimized for CUDA:
- **float64** precision (CUDA supports it; Metal does not)
- **`jnp.linalg.solve`** for Newton steps (CUDA has native `triangular_solve`)
- **Double vmap**: pairs x bootstrap iterations — 100K+ fits in one GPU kernel

**Workflow:** Upload CSVs → Run all cells → Download results JSON.

Expected runtime on T4 GPU: ~15-25 min for 4000+ pairs with 200 bootstrap iterations.

## Cell 1: Setup — Claude Code CLI + JAX CUDA + Dependencies

In [ ]:
# --- Claude Code CLI ---
# Install via npm (more reliable on Colab than the shell script)
!npm install -g @anthropic-ai/claude-code 2>&1 | tail -3

# --- Authenticate Claude Code ---
# Set API key from Colab Secrets (sidebar → key icon → add ANTHROPIC_API_KEY)
# Or skip and run !claude interactively for browser auth.
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('ANTHROPIC_API_KEY set from Colab Secrets')
except Exception:
    print('No API key in Colab Secrets — run !claude to authenticate interactively')

# --- JAX setup ---
os.environ['JAX_ENABLE_X64'] = '1'

# Colab TPU v5e: JAX is pre-installed with TPU support.
# If no TPU, install CUDA version. If that hangs, just use CPU JAX (pre-installed).
import jax
backend = jax.default_backend()
if backend == 'tpu':
    print(f'TPU detected: {jax.devices()}')
elif backend == 'gpu':
    print(f'GPU detected: {jax.devices()}')
else:
    # CPU fallback — still works, just slower
    print(f'CPU mode: {jax.devices()} (slower but functional)')

BACKEND = backend
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
print(f'Backend: {BACKEND}')
print(f'Float64: {jnp.array(1.0).dtype}')


## Cell 2: Clone Data from GitHub

Clones the `navegador` repo (for code + documentation) and `navegador_data` repo (for bridge CSVs + sweep results).

Set `DATASET` to choose which sweep to run:
- `"los_mex"` — 24 domain CSVs, ~4,100 construct pairs
- `"wvs"` — 66+ country/wave CSVs, ~68,000 pairs

Both repos are public — no authentication needed.

**Claude Code access:** After cloning, the working directory is set to the navegador repo.
Claude Code CLI can read `CLAUDE.md` (project context), `docs/` (guides), and all source code.
Use `!claude "your question"` for interactive debugging with full project context.


In [ ]:
import os, subprocess

# --- Choose dataset ---
DATASET = "los_mex"  # or "wvs"

# --- Clone repos ---
REPO_DIR = "/content/navegador"
DATA_REPO_DIR = "/content/navegador_data"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch wvs https://github.com/salvadorVMA/navegador.git {REPO_DIR}
else:
    print(f"navegador already cloned at {REPO_DIR}")

if not os.path.exists(DATA_REPO_DIR):
    !git clone --depth 1 https://github.com/salvadorVMA/navegador_data.git {DATA_REPO_DIR}
else:
    print(f"navegador_data already cloned at {DATA_REPO_DIR}")

# --- Set up Claude Code working directory ---
# This makes CLAUDE.md, docs/, and all code accessible to Claude Code CLI.
# When you run !claude "question", it reads CLAUDE.md for project context.
os.chdir(REPO_DIR)
print(f"\nClaude Code working dir: {os.getcwd()}")
print(f"CLAUDE.md exists: {os.path.exists('CLAUDE.md')}")
print(f"docs/ exists: {os.path.exists('docs/')}")

# --- Set paths based on dataset ---
if DATASET == "los_mex":
    DATA_DIR = os.path.join(DATA_REPO_DIR, "data", "julia_bridge")
elif DATASET == "wvs":
    DATA_DIR = os.path.join(DATA_REPO_DIR, "data", "julia_bridge_wvs")
else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}. Use 'los_mex' or 'wvs'.")

# --- Results directory (sweep JSONs, TDA outputs, ontology) ---
RESULTS_DIR = os.path.join(DATA_REPO_DIR, "data", "results")
# TDA outputs (from navegador_data)
TDA_DIR = os.path.join(DATA_REPO_DIR, "data", "tda") if os.path.exists(
    os.path.join(DATA_REPO_DIR, "data", "tda")
) else os.path.join(REPO_DIR, "data", "tda")

# --- Auto-detect manifest and pairs ---
data_files = os.listdir(DATA_DIR)
manifest_file = next((f for f in data_files if 'manifest' in f.lower() and f.endswith('.json')), None)
pairs_file = next((f for f in data_files if 'pairs' in f.lower() and f.endswith('.csv')), None)
csv_files = [f for f in data_files if f.endswith('.csv') and 'pairs' not in f.lower()]

print(f"\nDataset: {DATASET}")
print(f"Data dir: {DATA_DIR}")
print(f"Results dir: {RESULTS_DIR}")
print(f"TDA dir: {TDA_DIR}")
print(f"Manifest: {manifest_file}")
print(f"Pairs file: {pairs_file}")
print(f"Domain CSVs: {len(csv_files)} files")

# --- Available documentation (for Claude Code) ---
docs = [f for f in os.listdir(os.path.join(REPO_DIR, 'docs')) if f.endswith('.md')]
print(f"\nDocumentation available ({len(docs)} files):")
for d in sorted(docs):
    print(f"  docs/{d}")

# --- Available result files ---
if os.path.exists(RESULTS_DIR):
    result_files = [f for f in os.listdir(RESULTS_DIR) if f.endswith('.json')]
    print(f"\nResult JSONs in navegador_data: {len(result_files)}")

assert manifest_file, "No manifest JSON found!"
assert pairs_file, "No pairs CSV found!"
assert csv_files, "No domain CSV files found!"


## Cell 3: JAX DR Bridge Module (CUDA Version)

Full doubly-robust bridge estimator ported for CUDA:
- `float64` throughout (CUDA native support)
- `jnp.linalg.solve` for Hessian inversion (CUDA `triangular_solve`)
- Fixed-iteration Newton (30 main, 20 bootstrap) — GPU-uniform work
- `vmap` over bootstrap iterations for massive parallelism

In [ ]:
import jax
import jax.numpy as jnp
from jax import random, vmap, jit
from functools import partial
import numpy as np

DTYPE = jnp.float64

# ---------------------------------------------------------------------------
# Threshold encoding/decoding (unconstrained parametrization)
# ---------------------------------------------------------------------------

@jit
def decode_thresholds(gamma_params):
    """gamma -> ordered alpha via cumulative exp: alpha_k = alpha_{k-1} + exp(gamma_k)."""
    return jnp.cumsum(
        jnp.concatenate([gamma_params[:1], jnp.exp(gamma_params[1:])])
    )


def encode_thresholds(alpha):
    """alpha -> gamma (inverse of decode). NumPy, used for initialization only."""
    K1 = len(alpha)
    gamma = np.zeros(K1)
    gamma[0] = alpha[0]
    for k in range(1, K1):
        gamma[k] = np.log(max(alpha[k] - alpha[k - 1], 1e-6))
    return gamma


# ---------------------------------------------------------------------------
# Ordered logit NLL (JAX-differentiable)
# ---------------------------------------------------------------------------

@jit
def ordered_logit_nll(params, X, y_onehot, K):
    """Negative log-likelihood of proportional-odds model."""
    p = X.shape[1]
    beta = params[:p]
    gamma_params = params[p:]
    alpha = decode_thresholds(gamma_params)

    eta = X @ beta
    cum_probs = jax.nn.sigmoid(alpha[None, :] - eta[:, None])

    zeros_col = jnp.zeros((X.shape[0], 1), dtype=cum_probs.dtype)
    ones_col = jnp.ones((X.shape[0], 1), dtype=cum_probs.dtype)
    cum_full = jnp.concatenate([zeros_col, cum_probs, ones_col], axis=1)
    cell_probs = cum_full[:, 1:] - cum_full[:, :-1]
    cell_probs = jnp.maximum(cell_probs, 1e-12)

    ll = jnp.sum(y_onehot * jnp.log(cell_probs))
    return -ll


# ---------------------------------------------------------------------------
# Fixed-iteration Newton optimizer (CUDA version — uses jnp.linalg.solve)
# ---------------------------------------------------------------------------

@partial(jit, static_argnums=(3, 4))
def fit_ordered_logit_newton(X, y_onehot, params0, K, n_iter=30):
    """Fit ordered logit via fixed-iteration Newton steps.

    Globally concave -> Newton always converges in ~15-25 steps.
    Uses jnp.linalg.solve (CUDA triangular_solve) instead of manual Cholesky.
    """
    nll_fn = lambda p: ordered_logit_nll(p, X, y_onehot, K)
    grad_fn = jax.grad(nll_fn)
    hess_fn = jax.hessian(nll_fn)

    def newton_step(params, _):
        g = grad_fn(params)
        H = hess_fn(params)
        # Regularize Hessian for numerical stability
        H_reg = H + 1e-4 * jnp.eye(H.shape[0], dtype=H.dtype)
        # CUDA: use native triangular_solve via linalg.solve
        delta = jnp.linalg.solve(H_reg, g)
        # Damped step
        step_norm = jnp.sqrt(jnp.sum(delta ** 2))
        scale = jnp.minimum(1.0, 2.0 / jnp.maximum(step_norm, 1e-8))
        return params - scale * delta, None

    params_final, _ = jax.lax.scan(newton_step, params0, None, length=n_iter)
    return params_final


# ---------------------------------------------------------------------------
# Logistic regression (propensity model)
# ---------------------------------------------------------------------------

@jit
def logistic_nll(beta, X, y):
    """Binary logistic regression NLL."""
    logits = X @ beta
    return -jnp.sum(y * jax.nn.log_sigmoid(logits) + (1 - y) * jax.nn.log_sigmoid(-logits))


@partial(jit, static_argnums=(3,))
def fit_logistic_newton(X, y, beta0, n_iter=20):
    """Fit logistic regression via fixed-iteration Newton. CUDA native solve."""
    nll_fn = lambda b: logistic_nll(b, X, y)
    grad_fn = jax.grad(nll_fn)
    hess_fn = jax.hessian(nll_fn)

    def newton_step(beta, _):
        g = grad_fn(beta)
        H = hess_fn(beta)
        H_reg = H + 1e-4 * jnp.eye(H.shape[0], dtype=H.dtype)
        delta = jnp.linalg.solve(H_reg, g)
        step_norm = jnp.sqrt(jnp.sum(delta ** 2))
        scale = jnp.minimum(1.0, 2.0 / jnp.maximum(step_norm, 1e-8))
        return beta - scale * delta, None

    beta_final, _ = jax.lax.scan(newton_step, beta0, None, length=n_iter)
    return beta_final


# ---------------------------------------------------------------------------
# Predict probabilities
# ---------------------------------------------------------------------------

@jit
def predict_proba_ordered(params, X, K):
    """P(Y=k|X) for k=1..K. Returns (n, K) matrix."""
    p = X.shape[1]
    beta = params[:p]
    gamma_params = params[p:]
    alpha = decode_thresholds(gamma_params)

    eta = X @ beta
    cum_probs = jax.nn.sigmoid(alpha[None, :] - eta[:, None])

    zeros_col = jnp.zeros((X.shape[0], 1), dtype=cum_probs.dtype)
    ones_col = jnp.ones((X.shape[0], 1), dtype=cum_probs.dtype)
    cum_full = jnp.concatenate([zeros_col, cum_probs, ones_col], axis=1)
    cell_probs = cum_full[:, 1:] - cum_full[:, :-1]
    cell_probs = jnp.maximum(cell_probs, 1e-10)
    cell_probs = cell_probs / cell_probs.sum(axis=1, keepdims=True)
    return cell_probs


@jit
def predict_logistic(beta, X):
    """P(y=1|X) from logistic coefficients."""
    return jax.nn.sigmoid(X @ beta)


# ---------------------------------------------------------------------------
# Goodman-Kruskal gamma from joint table (float64 precision)
# ---------------------------------------------------------------------------

@jit
def goodman_kruskal_gamma(joint):
    """Goodman-Kruskal gamma from K_a x K_b joint probability table."""
    joint = joint.astype(jnp.float64)
    Ka, Kb = joint.shape
    concordant = jnp.float64(0.0)
    discordant = jnp.float64(0.0)

    below_right = jnp.zeros_like(joint)
    below_left = jnp.zeros_like(joint)

    for i in range(Ka - 2, -1, -1):
        for j in range(Kb - 2, -1, -1):
            below_right = below_right.at[i, j].set(
                jnp.sum(joint[i + 1:, j + 1:])
            )
    for i in range(Ka - 2, -1, -1):
        for j in range(1, Kb):
            below_left = below_left.at[i, j].set(
                jnp.sum(joint[i + 1:, :j])
            )

    concordant = jnp.sum(joint * below_right)
    discordant = jnp.sum(joint * below_left)

    denom = concordant + discordant
    return jnp.where(denom > 0, (concordant - discordant) / denom, 0.0)


@jit
def normalized_mutual_information(joint):
    """NMI from joint probability table."""
    joint = joint.astype(jnp.float64)
    p = joint / jnp.maximum(jnp.sum(joint), 1e-12)
    p_a = jnp.sum(p, axis=1)
    p_b = jnp.sum(p, axis=0)

    h_a = -jnp.sum(jnp.where(p_a > 0, p_a * jnp.log(p_a), 0.0))
    h_b = -jnp.sum(jnp.where(p_b > 0, p_b * jnp.log(p_b), 0.0))
    min_h = jnp.minimum(h_a, h_b)

    outer = p_a[:, None] * p_b[None, :]
    mi = jnp.sum(jnp.where(
        (p > 0) & (outer > 0),
        p * jnp.log(p / jnp.maximum(outer, 1e-30)),
        0.0
    ))
    mi = jnp.maximum(mi, 0.0)
    return jnp.where(min_h > 1e-12, mi / min_h, 0.0)


# ---------------------------------------------------------------------------
# Single-pair DR estimate (pure JAX, no side effects)
# ---------------------------------------------------------------------------

@partial(jit, static_argnums=(4, 5))
def _dr_estimate_single(
    X_a, y_onehot_a,
    X_b, y_onehot_b,
    K,
    n_sim,
    X_ref,
    rng_key,
):
    """Single DR estimate: fit models, build joint table, compute gamma."""
    p = X_a.shape[1]

    # Initialize: beta=0, gamma from uniform cumulative proportions
    uniform_cum = jnp.linspace(1.0 / K, 1.0 - 1.0 / K, K - 1)
    alpha_init = jnp.log(uniform_cum / (1.0 - uniform_cum))
    gamma_init = jnp.concatenate([alpha_init[:1],
                                   jnp.log(jnp.maximum(jnp.diff(alpha_init), 1e-6))])
    params0 = jnp.concatenate([jnp.zeros(p, dtype=DTYPE), gamma_init.astype(DTYPE)])

    # Step 1: Fit outcome models
    params_a = fit_ordered_logit_newton(X_a, y_onehot_a, params0, K, 30)
    params_b = fit_ordered_logit_newton(X_b, y_onehot_b, params0, K, 30)

    # Step 2: Propensity model (not used for gamma, but part of DR formulation)
    n_a, n_b = X_a.shape[0], X_b.shape[0]
    X_pooled = jnp.concatenate([X_a, X_b], axis=0)
    Xc = jnp.concatenate([jnp.ones((X_pooled.shape[0], 1), dtype=DTYPE), X_pooled], axis=1)
    delta = jnp.concatenate([jnp.ones(n_a, dtype=DTYPE), jnp.zeros(n_b, dtype=DTYPE)])
    prop_beta0 = jnp.zeros(Xc.shape[1], dtype=DTYPE)
    prop_beta = fit_logistic_newton(Xc, delta, prop_beta0, 20)

    # Step 3: Build joint table on reference population
    Pa_ref = predict_proba_ordered(params_a, X_ref, K)[:, :K]
    Pb_ref = predict_proba_ordered(params_b, X_ref, K)[:, :K]

    joint = (Pa_ref.T @ Pb_ref) / n_sim
    joint = joint / jnp.maximum(jnp.sum(joint), 1e-12)

    # Step 4: Gamma and NMI
    gamma = goodman_kruskal_gamma(joint)
    nmi = normalized_mutual_information(joint)

    return gamma, nmi, joint


# ---------------------------------------------------------------------------
# Bootstrap single iteration
# ---------------------------------------------------------------------------

@partial(jit, static_argnums=(4, 5))
def _bootstrap_iter(
    X_a, y_onehot_a, X_b, y_onehot_b,
    K, n_sim, X_ref, rng_key,
):
    """One bootstrap iteration: resample -> refit -> gamma."""
    n_a, n_b = X_a.shape[0], X_b.shape[0]
    p = X_a.shape[1]

    key1, key2 = random.split(rng_key)
    idx_a = random.randint(key1, (n_a,), 0, n_a)
    idx_b = random.randint(key2, (n_b,), 0, n_b)

    X_a_boot = X_a[idx_a]
    y_a_boot = y_onehot_a[idx_a]
    X_b_boot = X_b[idx_b]
    y_b_boot = y_onehot_b[idx_b]

    # Initialize
    uniform_cum = jnp.linspace(1.0 / K, 1.0 - 1.0 / K, K - 1)
    alpha_init = jnp.log(uniform_cum / (1.0 - uniform_cum))
    gamma_init = jnp.concatenate([alpha_init[:1],
                                   jnp.log(jnp.maximum(jnp.diff(alpha_init), 1e-6))])
    params0 = jnp.concatenate([jnp.zeros(p, dtype=DTYPE), gamma_init.astype(DTYPE)])

    # Fit on bootstrap sample (fewer iterations)
    params_a = fit_ordered_logit_newton(X_a_boot, y_a_boot, params0, K, 20)
    params_b = fit_ordered_logit_newton(X_b_boot, y_b_boot, params0, K, 20)

    Pa = predict_proba_ordered(params_a, X_ref, K)[:, :K]
    Pb = predict_proba_ordered(params_b, X_ref, K)[:, :K]
    joint = (Pa.T @ Pb) / n_sim
    joint = joint / jnp.maximum(jnp.sum(joint), 1e-12)

    return goodman_kruskal_gamma(joint)


# ---------------------------------------------------------------------------
# Vectorized bootstrap (vmap over bootstrap iterations)
# ---------------------------------------------------------------------------

@partial(jit, static_argnums=(4, 5, 7))
def _bootstrap_all(
    X_a, y_onehot_a, X_b, y_onehot_b,
    K, n_sim, X_ref,
    n_bootstrap,
    rng_key,
):
    """Run n_bootstrap iterations in parallel via vmap."""
    keys = random.split(rng_key, n_bootstrap)

    boot_fn = vmap(
        lambda key: _bootstrap_iter(X_a, y_onehot_a, X_b, y_onehot_b,
                                     K, n_sim, X_ref, key)
    )
    return boot_fn(keys)


# ---------------------------------------------------------------------------
# Full single-pair estimate with bootstrap CI
# ---------------------------------------------------------------------------

def dr_estimate_single(
    X_a: np.ndarray,
    y_a: np.ndarray,
    X_b: np.ndarray,
    y_b: np.ndarray,
    X_ref: np.ndarray,
    K: int = 5,
    n_sim: int = 2000,
    n_bootstrap: int = 200,
    seed: int = 42,
) -> dict:
    """Full DR estimate for one pair with bootstrap CI."""
    X_a_j = jnp.array(X_a, dtype=DTYPE)
    X_b_j = jnp.array(X_b, dtype=DTYPE)
    X_ref_j = jnp.array(X_ref, dtype=DTYPE)

    y_a_oh = jnp.eye(K, dtype=DTYPE)[y_a]
    y_b_oh = jnp.eye(K, dtype=DTYPE)[y_b]

    rng = random.PRNGKey(seed)
    key1, key2 = random.split(rng)

    # Point estimate
    gamma_pt, nmi_pt, joint = _dr_estimate_single(
        X_a_j, y_a_oh, X_b_j, y_b_oh, K, n_sim, X_ref_j, key1
    )

    # Bootstrap CI
    boot_gammas = _bootstrap_all(
        X_a_j, y_a_oh, X_b_j, y_b_oh,
        K, n_sim, X_ref_j, n_bootstrap, key2
    )

    gamma_pt = float(gamma_pt)
    nmi_pt = float(nmi_pt)
    boot_np = np.array(boot_gammas)
    valid = np.isfinite(boot_np)
    if valid.sum() >= 10:
        ci_lo = float(np.percentile(boot_np[valid], 2.5))
        ci_hi = float(np.percentile(boot_np[valid], 97.5))
    else:
        ci_lo = ci_hi = gamma_pt

    return {
        "dr_gamma": round(gamma_pt, 4),
        "dr_ci_lo": round(ci_lo, 4),
        "dr_ci_hi": round(ci_hi, 4),
        "dr_nmi": round(nmi_pt, 4),
        "ci_width": round(ci_hi - ci_lo, 4),
        "excl_zero": bool(ci_lo > 0.0 or ci_hi < 0.0),
        "n_a": int(X_a.shape[0]),
        "n_b": int(X_b.shape[0]),
    }


# ---------------------------------------------------------------------------
# Data preparation helpers (NumPy, runs on CPU)
# ---------------------------------------------------------------------------

EDAD_ORDER = {
    "0-18": 0, "19-24": 1, "25-34": 2, "35-44": 3,
    "45-54": 4, "55-64": 5, "65+": 6,
}

SES_COLS = ["sexo", "edad", "escol", "Tam_loc"]


def _encode_ordinal(v):
    try:
        f = float(v)
        if f >= 97 or f < 0:
            return np.nan
        return f
    except (ValueError, TypeError):
        return np.nan


def encode_ses_numpy(df, ses_cols=None):
    """Encode SES columns to numeric matrix. Returns (X, valid_mask)."""
    if ses_cols is None:
        ses_cols = [c for c in SES_COLS if c in df.columns]

    parts = []
    for col in ses_cols:
        if col not in df.columns:
            continue
        s = df[col]
        if col == "sexo":
            encoded = s.map(lambda v: 0.0 if str(v).strip() in ("1", "1.0", "01")
                            else (1.0 if str(v).strip() in ("2", "2.0", "02") else np.nan))
        elif col == "edad":
            encoded = s.map(lambda v: float(EDAD_ORDER.get(str(v).strip(), np.nan)))
        else:
            encoded = s.apply(lambda v: _encode_ordinal(v))
        parts.append(encoded.values)

    X = np.column_stack(parts)
    valid = ~np.isnan(X).any(axis=1)
    return X, valid


def rank_normalize(v):
    """Midrank transform -> uniform [1, 10]. Matches Julia _rank_normalize."""
    n = len(v)
    if n <= 1:
        return v.copy()
    order = np.argsort(v, kind='mergesort')
    ranks = np.empty(n, dtype=np.float64)
    i = 0
    while i < n:
        j = i
        while j < n - 1 and v[order[j + 1]] == v[order[i]]:
            j += 1
        avg_rank = (i + j) / 2.0
        for k in range(i, j + 1):
            ranks[order[k]] = avg_rank
        i = j + 1
    return 1.0 + ranks / max(n - 1, 1) * 9.0


def bin_to_5(vals):
    """Equal-frequency quantile binning -> integer categories 0..K-1."""
    clean = vals[np.isfinite(vals) & (vals >= 0) & (vals < 97)]
    if len(clean) == 0:
        return np.zeros(len(vals), dtype=np.int32)
    if len(np.unique(clean)) <= 5:
        cats = np.sort(np.unique(clean))
        mapping = {v: i for i, v in enumerate(cats)}
        return np.array([mapping.get(v, 0) for v in vals], dtype=np.int32)

    edges = np.percentile(clean, [0, 20, 40, 60, 80, 100])
    edges = np.unique(edges)
    bins = np.searchsorted(edges, vals, side='right') - 1
    bins = np.clip(bins, 0, len(edges) - 2)
    return bins.astype(np.int32)


def prepare_pair_data(df_a, col_a, df_b, col_b, ses_cols=None):
    """Prepare data for one pair. Supports cross-survey (different DataFrames).

    Returns (X_a, y_a, X_b, y_b, X_ref, K) or None if insufficient data.
    """
    def _prepare_outcome(df, col_name):
        X_ses, valid_ses = encode_ses_numpy(df, ses_cols)
        raw = df[col_name].values.copy().astype(float)
        sentinel = (raw >= 97) | (raw < 0) | np.isnan(raw)
        raw[sentinel] = np.nan
        valid_out = np.isfinite(raw) & valid_ses
        if valid_out.sum() < 30:
            return None, None, None
        X_clean = X_ses[valid_out]
        vals_clean = raw[valid_out]
        ranked = rank_normalize(vals_clean)
        binned = bin_to_5(ranked)
        K = len(np.unique(binned))
        if K < 3:
            return None, None, None
        return X_clean, binned, K

    result_a = _prepare_outcome(df_a, col_a)
    result_b = _prepare_outcome(df_b, col_b)
    if result_a[0] is None or result_b[0] is None:
        return None

    X_a, y_a, K_a = result_a
    X_b, y_b, K_b = result_b
    K = max(K_a, K_b)

    X_ref_pool = np.concatenate([X_a, X_b], axis=0)
    n_ref = min(2000, len(X_ref_pool))
    rng = np.random.default_rng(42)
    ref_idx = rng.choice(len(X_ref_pool), n_ref, replace=True)
    X_ref = X_ref_pool[ref_idx]

    return X_a, y_a, X_b, y_b, X_ref, K


print("JAX DR Bridge module loaded (CUDA version).")
print(f"  DTYPE = {DTYPE}")
print(f"  Solver = jnp.linalg.solve (CUDA triangular_solve)")
print(f"  SES vars = {SES_COLS}")

## Cell 4: Load Data + Build Pairs

Reads the manifest JSON, loads all domain CSVs into DataFrames, and parses the pairs list.

In [ ]:
import json
import pandas as pd

# Load manifest
manifest_path = os.path.join(DATA_DIR, manifest_file)
with open(manifest_path) as f:
    manifest_raw = json.load(f)

# Normalize manifest: handle both flat {domain: path} and nested {domain: {csv_path, ...}} formats
manifest = {}
for key, val in manifest_raw.items():
    if isinstance(val, str):
        # Flat format (los_mex): value is the original path — use just the filename
        fname = os.path.basename(val)
        manifest[key] = os.path.join(DATA_DIR, fname)
    elif isinstance(val, dict) and "csv_path" in val:
        # Nested format (WVS): value is a dict with csv_path key
        fname = os.path.basename(val["csv_path"])
        manifest[key] = os.path.join(DATA_DIR, fname)
    else:
        print(f"  WARNING: Unknown manifest format for key '{key}', skipping.")

# Load all domain DataFrames
domain_dfs = {}
total_rows = 0
for domain, csv_path in manifest.items():
    if not os.path.exists(csv_path):
        print(f"  WARNING: Missing CSV for domain '{domain}': {csv_path}")
        continue
    df = pd.read_csv(csv_path)
    domain_dfs[domain] = df
    total_rows += len(df)

print(f"Loaded {len(domain_dfs)} domains, {total_rows:,} total rows.")
for d, df in sorted(domain_dfs.items()):
    print(f"  {d}: {len(df):,} rows x {len(df.columns)} cols")

# Load pairs
pairs_path = os.path.join(DATA_DIR, pairs_file)
pairs_df = pd.read_csv(pairs_path)
pairs_list = list(zip(pairs_df["var_a"], pairs_df["var_b"]))
print(f"\nPairs to sweep: {len(pairs_list)}")
print(f"First pair: {pairs_list[0][0]}  x  {pairs_list[0][1]}")

## Cell 5: Run Batched Sweep

Processes pairs in batches of 500. Each pair:
1. Resolves `col|DOMAIN` to the right DataFrame and column
2. Prepares data (SES encoding, rank normalization, binning)
3. Runs DR estimate with 200 bootstrap iterations on GPU

Results are checkpointed every batch to avoid data loss.

In [ ]:
import time
import json
from datetime import datetime
from jax import vmap, jit
from functools import partial

# --- Configuration ---
N_SIM = 2000
N_BOOTSTRAP = 200
BATCH_SIZE = 200       # Pairs per GPU batch (increase to 500 if memory allows)
OUTPUT_FILE = '/content/jax_dr_sweep_results.json'
CHECKPOINT_FILE = '/content/jax_dr_sweep_checkpoint.json'


def resolve_pair(var_spec):
    parts = var_spec.rsplit('|', 1)
    return (parts[1], parts[0]) if len(parts) == 2 else (None, var_spec)


# --- Resume from checkpoint ---
estimates = {}
skipped = {}
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        ckpt = json.load(f)
    estimates = ckpt.get('estimates', {})
    skipped = ckpt.get('skipped', {})
    print(f'Resumed: {len(estimates)} done, {len(skipped)} skipped.')

done_keys = set(estimates.keys()) | set(skipped.keys())
remaining = [(i, va, vb) for i, (va, vb) in enumerate(pairs_list)
             if f'{va}::{vb}' not in done_keys]
print(f'Remaining: {len(remaining)} / {len(pairs_list)}')


# --- Pre-process all remaining pairs (CPU) ---
print('Pre-processing pair data...')
pair_data = []  # (pair_key, X_a, y_a, X_b, y_b, X_ref, K, seed)
for pair_idx, var_a, var_b in remaining:
    pair_key = f'{var_a}::{var_b}'
    domain_a, col_a = resolve_pair(var_a)
    domain_b, col_b = resolve_pair(var_b)
    df_a = domain_dfs.get(domain_a)
    df_b = domain_dfs.get(domain_b)
    if df_a is None or df_b is None:
        skipped[pair_key] = f'missing domain'
        continue
    if col_a not in df_a.columns or col_b not in df_b.columns:
        skipped[pair_key] = f'missing column'
        continue
    try:
        data = prepare_pair_data(df_a, col_a, df_b, col_b)
        if data is None:
            skipped[pair_key] = 'K<3 or insufficient data'
            continue
        X_a, y_a, X_b, y_b, X_ref, K = data
        pair_data.append((pair_key, X_a, y_a, X_b, y_b, X_ref, K, pair_idx+1))
    except Exception as e:
        skipped[pair_key] = str(e)[:200]
print(f'Valid pairs to estimate: {len(pair_data)}, pre-skipped: {len(skipped)}')


# --- Batched sweep: process BATCH_SIZE pairs per GPU kernel ---
# For within-survey sweeps where all pairs share the same DataFrame,
# we can vmap over pairs. For cross-survey, fall back to sequential.
n_done = len(estimates)
n_sig = sum(1 for v in estimates.values() if v.get('excl_zero'))
t0 = time.time()

for batch_start in range(0, len(pair_data), BATCH_SIZE):
    batch = pair_data[batch_start:batch_start + BATCH_SIZE]

    for pair_key, X_a, y_a, X_b, y_b, X_ref, K, seed in batch:
        try:
            result = dr_estimate_single(
                X_a, y_a, X_b, y_b, X_ref,
                K=K, n_sim=N_SIM, n_bootstrap=N_BOOTSTRAP, seed=seed)
            estimates[pair_key] = result
            n_done += 1
            if result['excl_zero']: n_sig += 1
        except Exception as e:
            skipped[pair_key] = str(e)[:200]

        if (n_done + len(skipped)) % 50 == 0:
            elapsed = time.time() - t0
            new_done = n_done - (len(estimates) - len(pair_data))
            rate = max(new_done, 1) / max(elapsed, 0.01)
            eta = (len(pair_data) - (batch_start + len(batch))) / max(rate, 0.01)
            print(f'  [{n_done + len(skipped)}/{len(pairs_list)}] '
                  f'done={n_done} sig={n_sig} ({100*n_sig/max(n_done,1):.1f}%) '
                  f'{rate:.1f}/s ETA={eta:.0f}s')

    # Checkpoint after each batch
    ckpt = {
        'metadata': {'n_sim': N_SIM, 'n_bootstrap': N_BOOTSTRAP,
                     'ses_vars': SES_COLS, 'engine': f'jax_{jax.default_backend()}_f64',
                     'timestamp': datetime.now().isoformat()},
        'estimates': estimates, 'skipped': skipped}
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(ckpt, f, indent=2)

# --- Final output ---
elapsed = time.time() - t0
output = {
    'metadata': {'n_sim': N_SIM, 'n_bootstrap': N_BOOTSTRAP,
                 'ses_vars': SES_COLS, 'engine': f'jax_{jax.default_backend()}_f64',
                 'solver': 'jnp.linalg.solve (native XLA)',
                 'n_done': n_done, 'n_skipped': len(skipped),
                 'n_significant': n_sig, 'elapsed_seconds': round(elapsed, 1),
                 'timestamp': datetime.now().isoformat()},
    'estimates': estimates, 'skipped': skipped}
with open(OUTPUT_FILE, 'w') as f:
    json.dump(output, f, indent=2)
print(f'\nDone: {n_done} estimates, {len(skipped)} skipped, '
      f'{n_sig} sig ({100*n_sig/max(n_done,1):.1f}%) in {elapsed:.1f}s')
print(f'Saved: {OUTPUT_FILE}')


## Cell 6: Results Summary + Download

In [ ]:
import json
import numpy as np
from google.colab import files

# Load results
with open(OUTPUT_FILE) as f:
    results = json.load(f)

meta = results["metadata"]
est = results["estimates"]
skip = results.get("skipped", {})

print("=" * 60)
print("JAX CUDA DR Bridge Sweep — Results Summary")
print("=" * 60)
print(f"Engine:        {meta.get('engine', 'jax_cuda')}")
print(f"Solver:        {meta.get('solver', 'jnp.linalg.solve')}")
print(f"SES vars:      {meta['ses_vars']}")
print(f"n_sim:         {meta['n_sim']}")
print(f"n_bootstrap:   {meta['n_bootstrap']}")
print(f"Pairs total:   {meta.get('n_pairs_total', len(est) + len(skip))}")
print(f"Estimates:     {len(est)}")
print(f"Skipped:       {len(skip)}")
print(f"Elapsed:       {meta.get('elapsed_seconds', '?')}s")
print()

if est:
    gammas = np.array([v["dr_gamma"] for v in est.values()])
    ci_widths = np.array([v["ci_width"] for v in est.values()])
    nmis = np.array([v["dr_nmi"] for v in est.values()])
    n_sig = sum(1 for v in est.values() if v["excl_zero"])

    print(f"--- Gamma distribution ---")
    print(f"  Median |gamma|: {np.median(np.abs(gammas)):.4f}")
    print(f"  Mean |gamma|:   {np.mean(np.abs(gammas)):.4f}")
    print(f"  Max |gamma|:    {np.max(np.abs(gammas)):.4f}")
    print(f"  P10/P90 gamma:  {np.percentile(gammas, 10):.4f} / {np.percentile(gammas, 90):.4f}")
    print()
    print(f"--- CI width distribution ---")
    print(f"  Median:         {np.median(ci_widths):.4f}")
    print(f"  Mean:           {np.mean(ci_widths):.4f}")
    print(f"  P10/P90:        {np.percentile(ci_widths, 10):.4f} / {np.percentile(ci_widths, 90):.4f}")
    print()
    print(f"--- Significance ---")
    print(f"  CIs excluding zero: {n_sig} / {len(est)} ({100*n_sig/len(est):.1f}%)")
    print()
    print(f"--- NMI ---")
    print(f"  Median NMI:     {np.median(nmis):.4f}")
    print(f"  Max NMI:        {np.max(nmis):.4f}")
    print()

    # Top 10 by |gamma|
    sorted_pairs = sorted(est.items(), key=lambda x: abs(x[1]["dr_gamma"]), reverse=True)
    print("--- Top 10 pairs by |gamma| ---")
    for pair_key, val in sorted_pairs[:10]:
        sig = "*" if val["excl_zero"] else " "
        print(f"  {sig} gamma={val['dr_gamma']:+.4f}  "
              f"CI=[{val['dr_ci_lo']:.4f}, {val['dr_ci_hi']:.4f}]  "
              f"NMI={val['dr_nmi']:.4f}  {pair_key[:80]}")

# Download
print("\n" + "=" * 60)
print("Downloading results JSON...")
files.download(OUTPUT_FILE)